In [1]:
import time

from lerobot.processor import make_default_processors

from lerobot.teleoperators.so101_leader.config_so101_leader import SO101LeaderConfig
from lerobot.teleoperators.so101_leader.so101_leader import SO101Leader
from lerobot.teleoperators.bi_so100_leader import BiSO100Leader, BiSO100LeaderConfig

from lerobot.utils.robot_utils import busy_wait
from lerobot.utils.visualization_utils import init_rerun, log_rerun_data
from lerobot.robots.xlerobot import XLerobotClientConfig, XLerobotClient
from lerobot.cameras.configs import CameraConfig, Cv2Rotation, ColorMode
from lerobot.cameras.realsense import RealSenseCamera, RealSenseCameraConfig

FPS = 30

In [14]:
camera_config = {
            "head": RealSenseCameraConfig(
            serial_number_or_name="141722076677",  # Replace with camera SN
            fps=30,
            width=1280,
            height=720,
            color_mode=ColorMode.BGR, # Request BGR output
            rotation=Cv2Rotation.NO_ROTATION,
            use_depth=False
        )
}

# Initialize the robot and teleoperator config

follower_config = XLerobotClientConfig(remote_ip = '10.16.116.5', cameras = camera_config)

# Initialize the robot and teleoperator
follower = XLerobotClient(follower_config)

# Connect to the robot and teleoperator
follower.connect()

In [4]:
leader_config = BiSO100LeaderConfig(left_arm_port="/dev/ttyACM0", right_arm_port= "/dev/ttyACM1", id="leader_arm")
leader = BiSO100Leader(leader_config)
leader.connect()

In [4]:
def transform_arm_keys(original_dict: dict) -> dict:
    """
    Преобразует ключи формата left_* и right_* в left_arm_* и right_arm_*
    """
    transformed = {}
    
    for key, value in original_dict.items():
        if key.startswith('left_'):
            new_key = key.replace('left_', 'left_arm_', 1)
        elif key.startswith('right_'):
            new_key = key.replace('right_', 'right_arm_', 1)
        else:
            new_key = key  # Оставляем без изменений
            
        transformed[new_key] = value
    
    return transformed

In [6]:
leader.get_action()

{'left_shoulder_pan.pos': 4.073394495412842,
 'left_shoulder_lift.pos': -1.1149228130360171,
 'left_elbow_flex.pos': 9.12408759124088,
 'left_wrist_flex.pos': 4.823428079242035,
 'left_wrist_roll.pos': -3.950880939668977,
 'left_gripper.pos': 0.1654259718775848,
 'right_shoulder_pan.pos': -8.012126461671727,
 'right_shoulder_lift.pos': -91.30434782608695,
 'right_elbow_flex.pos': 100.0,
 'right_wrist_flex.pos': 42.39086087311301,
 'right_wrist_roll.pos': -14.737674663854477,
 'right_gripper.pos': 0.49464138499587795}

In [15]:
follower.get_observation()

{'left_arm_shoulder_pan.pos': -4.515173945225754,
 'left_arm_shoulder_lift.pos': -2.4828767123287605,
 'left_arm_elbow_flex.pos': 58.997722095671975,
 'left_arm_wrist_flex.pos': 6.833407771326478,
 'left_arm_wrist_roll.pos': -1.5319598520866435,
 'left_arm_gripper.pos': 0.48543689320388345,
 'right_arm_shoulder_pan.pos': -8.09013668267454,
 'right_arm_shoulder_lift.pos': -98.4816533108393,
 'right_arm_elbow_flex.pos': 70.05494505494505,
 'right_arm_wrist_flex.pos': 88.72148084373654,
 'right_arm_wrist_roll.pos': -14.694960212201593,
 'right_arm_gripper.pos': 0.8333333333333334,
 'head_motor_1.pos': -0.20964360587002773,
 'head_motor_2.pos': -28.149606299212607,
 'x.vel': 0.0,
 'y.vel': 0.0,
 'theta.vel': 0.0,
 'observation.state': array([ -4.515174 ,  -2.4828768,  58.997723 ,   6.833408 ,  -1.5319599,
          0.4854369,  -8.090137 , -98.48165  ,  70.05495  ,  88.72148  ,
        -14.694961 ,   0.8333333,  -0.2096436, -28.149607 ,   0.       ,
          0.       ,   0.       ], dtype=

In [13]:
follower.disconnect()

In [20]:
obs = follower.get_observation()
del obs['observation.state']
del obs['head']
obs

{'left_arm_shoulder_pan.pos': -4.515173945225754,
 'left_arm_shoulder_lift.pos': -1.9691780821917746,
 'left_arm_elbow_flex.pos': 57.8132118451025,
 'left_arm_wrist_flex.pos': 6.654756587762407,
 'left_arm_wrist_roll.pos': -1.6376122556788175,
 'left_arm_gripper.pos': 0.48543689320388345,
 'right_arm_shoulder_pan.pos': -8.09013668267454,
 'right_arm_shoulder_lift.pos': -98.4816533108393,
 'right_arm_elbow_flex.pos': 70.05494505494505,
 'right_arm_wrist_flex.pos': 88.72148084373654,
 'right_arm_wrist_roll.pos': -14.748010610079575,
 'right_arm_gripper.pos': 0.8333333333333334,
 'head_motor_1.pos': -0.20964360587002773,
 'head_motor_2.pos': -28.346456692913392,
 'x.vel': 0.0,
 'y.vel': 0.0,
 'theta.vel': 0.0}

In [40]:
obs['head_motor_1.pos'] = 110
obs['x.vel'] = 0
obs['y.vel'] = 0
obs['theta.vel'] = 0

follower.send_action(obs)

{'left_arm_shoulder_pan.pos': np.float32(-4.515174),
 'left_arm_shoulder_lift.pos': np.float32(-1.9691781),
 'left_arm_elbow_flex.pos': np.float32(57.813213),
 'left_arm_wrist_flex.pos': np.float32(6.6547565),
 'left_arm_wrist_roll.pos': np.float32(-1.6376122),
 'left_arm_gripper.pos': np.float32(0.4854369),
 'right_arm_shoulder_pan.pos': np.float32(-8.090137),
 'right_arm_shoulder_lift.pos': np.float32(-98.48165),
 'right_arm_elbow_flex.pos': np.float32(70.05495),
 'right_arm_wrist_flex.pos': np.float32(88.72148),
 'right_arm_wrist_roll.pos': np.float32(-14.748011),
 'right_arm_gripper.pos': np.float32(0.8333333),
 'head_motor_1.pos': np.float32(110.0),
 'head_motor_2.pos': np.float32(-30.0),
 'x.vel': np.float32(0.0),
 'y.vel': np.float32(0.0),
 'theta.vel': np.float32(0.0),
 'action': array([ -4.515174 ,  -1.9691781,  57.813213 ,   6.6547565,  -1.6376122,
          0.4854369,  -8.090137 , -98.48165  ,  70.05495  ,  88.72148  ,
        -14.748011 ,   0.8333333, 110.       , -30.     

In [64]:
follower.send_action(obs)

{'left_arm_shoulder_pan.pos': np.float32(-4.515174),
 'left_arm_shoulder_lift.pos': np.float32(-83.13356),
 'left_arm_elbow_flex.pos': np.float32(99.54442),
 'left_arm_wrist_flex.pos': np.float32(33.80974),
 'left_arm_wrist_roll.pos': np.float32(1.848917),
 'left_arm_gripper.pos': np.float32(33.35645),
 'right_arm_shoulder_pan.pos': np.float32(-8.16402),
 'right_arm_shoulder_lift.pos': np.float32(-98.48165),
 'right_arm_elbow_flex.pos': np.float32(70.05495),
 'right_arm_wrist_flex.pos': np.float32(88.72148),
 'right_arm_wrist_roll.pos': np.float32(-14.694961),
 'right_arm_gripper.pos': np.float32(0.8333333),
 'head_motor_1.pos': np.float32(150.0),
 'head_motor_2.pos': np.float32(-23.818897),
 'x.vel': np.float32(0.0),
 'y.vel': np.float32(0.0),
 'theta.vel': np.float32(0.0),
 'action': array([ -4.515174 , -83.13356  ,  99.54442  ,  33.80974  ,   1.848917 ,
         33.35645  ,  -8.16402  , -98.48165  ,  70.05495  ,  88.72148  ,
        -14.694961 ,   0.8333333, 150.       , -23.818897 

In [4]:
follower.disconnect()

In [17]:
action = leader.get_action()
action = {f"left_arm_{k}": v for k, v in action.items()}
action["head_motor_1.pos"] = 0.
action["head_motor_2.pos"] = 0.
follower.send_action(action)

{'left_arm_shoulder_pan.pos': np.float32(-6.495413),
 'left_arm_shoulder_lift.pos': np.float32(-100.0),
 'left_arm_elbow_flex.pos': np.float32(96.076645),
 'left_arm_wrist_flex.pos': np.float32(71.834625),
 'left_arm_wrist_roll.pos': np.float32(-5.65937),
 'left_arm_gripper.pos': np.float32(29.528536),
 'head_motor_1.pos': np.float32(0.0),
 'head_motor_2.pos': np.float32(0.0),
 'action': array([  -6.495413, -100.      ,   96.076645,   71.834625,   -5.65937 ,
          29.528536,    0.      ,    0.      ], dtype=float32)}

In [14]:
follower.send_action(action)

{'left_arm_shoulder_pan.pos': np.float32(-6.495413),
 'left_arm_shoulder_lift.pos': np.float32(-100.0),
 'left_arm_elbow_flex.pos': np.float32(95.89416),
 'left_arm_wrist_flex.pos': np.float32(72.35142),
 'left_arm_wrist_roll.pos': np.float32(-5.7127604),
 'left_arm_gripper.pos': np.float32(21.009098),
 'head_motor_1.pos': np.float32(0.0),
 'head_motor_2.pos': np.float32(0.0),
 'action': array([  -6.495413 , -100.       ,   95.89416  ,   72.35142  ,
          -5.7127604,   21.009098 ,    0.       ,    0.       ],
       dtype=float32)}

In [5]:
follower.disconnect()

### Phone server

In [2]:
from lerobot.teleoperators.keyboard.teleop_keyboard import KeyboardTeleop, KeyboardTeleopConfig
from phone_server import PhoneServer
import numpy as np
from time import sleep

In [3]:
keyboard_config = KeyboardTeleopConfig()
keyboard = KeyboardTeleop(keyboard_config)
keyboard.connect()

headset_server = PhoneServer()
headset_server.run()

Server started in background thread

XLeRobotHead - IMU & RealSense Server
✓ Server started with HTTPS support!

📱 Mobile Device: Open browser and go to:
   https://10.16.116.223:8443

   • iOS: Use Safari
   • Android: Use Chrome or Firefox

Features available:
  • Video Frame Streaming (external frames)
  • Mobile IMU/Orientation Data (iOS & Android)
Waiting for connection...



Video stream started for 10.16.120.85


ERROR:aiohttp.server:Error handling request from 10.16.120.85
Traceback (most recent call last):
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/web_protocol.py", line 510, in _handle_request
    resp = await request_handler(request)
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/web_app.py", line 569, in _handle
    return await handler(request)
  File "/home/erkhovartem/lerobot/examples/phone_server.py", line 865, in realsense_stream_handler
    await response.write(b'--frame\r\n')
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/web_response.py", line 560, in write
    await self._payload_writer.write(data)
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/http_writer.py", line 202, in write
    self._write_chunked_payload(chunk)
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/http_writer.py",

Video stream started for 10.16.120.85

Phone connected from 10.16.120.85

✓ Calibration set: Roll=-80.30°, Pitch=0.90°, Yaw=0.10°
[Normalized] Roll:    6.90° | Pitch:    0.40° | Yaw:   -3.10°

ERROR:aiohttp.server:Error handling request from 10.16.120.85
Traceback (most recent call last):
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/web_protocol.py", line 510, in _handle_request
    resp = await request_handler(request)
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/web_app.py", line 569, in _handle
    return await handler(request)
  File "/home/erkhovartem/lerobot/examples/phone_server.py", line 865, in realsense_stream_handler
    await response.write(b'--frame\r\n')
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/web_response.py", line 560, in write
    await self._payload_writer.write(data)
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/http_writer.py", line 202, in write
    self._write_chunked_payload(chunk)
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/http_writer.py",

...

Video stream started for 10.16.120.85


ERROR:aiohttp.server:Error handling request from 10.16.120.85
Traceback (most recent call last):
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/web_protocol.py", line 510, in _handle_request
    resp = await request_handler(request)
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/web_app.py", line 569, in _handle
    return await handler(request)
  File "/home/erkhovartem/lerobot/examples/phone_server.py", line 865, in realsense_stream_handler
    await response.write(b'--frame\r\n')
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/web_response.py", line 560, in write
    await self._payload_writer.write(data)
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/http_writer.py", line 202, in write
    self._write_chunked_payload(chunk)
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/http_writer.py",

Video stream started for 10.16.120.85


ERROR:aiohttp.server:Error handling request from 10.16.120.85
Traceback (most recent call last):
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/web_protocol.py", line 510, in _handle_request
    resp = await request_handler(request)
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/web_app.py", line 569, in _handle
    return await handler(request)
  File "/home/erkhovartem/lerobot/examples/phone_server.py", line 865, in realsense_stream_handler
    await response.write(b'--frame\r\n')
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/web_response.py", line 560, in write
    await self._payload_writer.write(data)
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/http_writer.py", line 202, in write
    self._write_chunked_payload(chunk)
  File "/home/erkhovartem/miniconda3/envs/lerobot/lib/python3.10/site-packages/aiohttp/http_writer.py",

Video stream started for 10.16.120.85


In [6]:
while(True):
    pressed_keys = set(keyboard.get_action().keys())
    keyboard_keys = np.array(list(pressed_keys))
    if 'c' in pressed_keys:
        headset_server.recalibrate()
    print(headset_server.get_angles()['roll'])
    sleep(0.1)

-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000000001
-76.60000000

KeyboardInterrupt: 

### PedaL teleop

In [1]:
from xlerobot_pedal_teleop import XlerobotPedalTeleop
import time
import numpy as np

In [4]:
teleop = XlerobotPedalTeleop()
teleop.connect()

In [ ]:
while(True):
    action = teleop.get_action()
    print(action)
    time.sleep(0.1)
